In [9]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score
)
from sklearn.pipeline import Pipeline


# 1. Read training data
X_train = pd.read_csv("../data/model_inputs/X_train.csv")

y_train_feasibility = pd.read_csv(
    "../data/model_inputs/y_train_feasibility.csv"
)["feasibility_binary"]

y_train_willingness = pd.read_csv(
    "../data/model_inputs/y_train_willingness.csv"
)["willingness_binary"]

os.makedirs("../results", exist_ok=True)

In [10]:
# 2. Define 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. Define parameter search space
param_distributions = {
    "rf__n_estimators": [300, 500, 700, 1000],
    "rf__max_depth": [None, 10, 20, 30, 40],
    "rf__min_samples_split": [2, 5, 10, 20],
    "rf__min_samples_leaf": [1, 2, 4, 8],
    "rf__max_features": ["sqrt", "log2", 0.5, 0.75],
    "rf__class_weight": [None, "balanced"]
}

In [11]:
# 4. Tune and evaluate Random Forest
def run_rf_model(X, y, outcome_name):

    rf_base = RandomForestClassifier(
        random_state=42,
        n_jobs=1
    )

    rf_model = Pipeline([
        ("rf", rf_base)
    ])

    rf_search = RandomizedSearchCV(
        estimator=rf_model,
        param_distributions=param_distributions,
        n_iter=1000,
        scoring="roc_auc",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1,
        return_train_score=False
    )

    rf_search.fit(X, y)

    clean_best_params = {
        key.replace("rf__", ""): value
        for key, value in rf_search.best_params_.items()
    }

    scores = {
        "AUC": [],
        "Precision": [],
        "Recall": [],
        "Accuracy": [],
        "F1": []
    }

    for train_idx, val_idx in cv.split(X, y):

        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]

        X_val_fold = X.iloc[val_idx]
        y_val_fold = y.iloc[val_idx]

        fold_rf = RandomForestClassifier(
            **clean_best_params,
            random_state=42,
            n_jobs=1
        )

        fold_model = Pipeline([
            ("rf", fold_rf)
        ])

        fold_model.fit(X_train_fold, y_train_fold)

        y_val_pred = fold_model.predict(X_val_fold)
        y_val_prob = fold_model.predict_proba(
            X_val_fold
        )[:, 1]


        scores["AUC"].append(
            roc_auc_score(y_val_fold, y_val_prob)
        )

        scores["Precision"].append(
            precision_score(
                y_val_fold,
                y_val_pred,
                zero_division=0
            )
        )

        scores["Recall"].append(
            recall_score(
                y_val_fold,
                y_val_pred,
                zero_division=0
            )
        )

        scores["Accuracy"].append(
            accuracy_score(
                y_val_fold,
                y_val_pred
            )
        )

        scores["F1"].append(
            f1_score(
                y_val_fold,
                y_val_pred,
                zero_division=0
            )
        )

    metric_order = [
        "AUC",
        "Precision",
        "Recall",
        "Accuracy",
        "F1"
    ]

    cv_summary = pd.DataFrame({
        "Metric": metric_order,
        "CV Score (SE)": [
            (
                f"{np.mean(scores[metric]):.3f} "
                f"({np.std(scores[metric], ddof=1) / np.sqrt(5):.3f})"
            )
            for metric in metric_order
        ]
    })

    print(f"\n{outcome_name} Random Forest")

    print("\nBest parameters:")
    print(clean_best_params)

    print(f"\n{outcome_name} Random Forest: 5-fold CV")
    print(cv_summary.to_string(index=False))

    return {
        "best_params": clean_best_params,
        "cv_summary": cv_summary
    }

In [12]:
# 5. Feasibility Random Forest
feasibility_rf_results = run_rf_model(
    X=X_train,
    y=y_train_feasibility,
    outcome_name="Feasibility"
)

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits

Feasibility Random Forest

Best parameters:
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 10, 'class_weight': None}

Feasibility Random Forest: 5-fold CV
   Metric CV Score (SE)
      AUC 0.636 (0.007)
Precision 0.774 (0.000)
   Recall 0.997 (0.001)
 Accuracy 0.773 (0.001)
       F1 0.871 (0.000)


In [13]:
# Save results
joblib.dump(
    feasibility_rf_results,
    "../results/rf_feasibility_results.joblib"
)

feasibility_rf_results["cv_summary"].to_csv(
    "../results/rf_feasibility_cv_summary.csv",
    index=False
)

In [14]:
# 6. Willingness Random Forest
willingness_rf_results = run_rf_model(
    X=X_train,
    y=y_train_willingness,
    outcome_name="Willingness"
)

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits

Willingness Random Forest

Best parameters:
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None, 'class_weight': None}

Willingness Random Forest: 5-fold CV
   Metric CV Score (SE)
      AUC 0.710 (0.011)
Precision 0.856 (0.001)
   Recall 0.998 (0.001)
 Accuracy 0.854 (0.001)
       F1 0.921 (0.000)


In [15]:
# Save results
joblib.dump(
    willingness_rf_results,
    "../results/rf_willingness_results.joblib"
)

willingness_rf_results["cv_summary"].to_csv(
    "../results/rf_willingness_cv_summary.csv",
    index=False
)

In [16]:
# 7. Read independent test data
X_test = pd.read_csv("../data/model_inputs/X_test.csv")

y_test_feasibility = pd.read_csv(
    "../data/model_inputs/y_test_feasibility.csv"
)["feasibility_binary"]

y_test_willingness = pd.read_csv(
    "../data/model_inputs/y_test_willingness.csv"
)["willingness_binary"]


# 8. Evaluate independent test set
def evaluate_rf_test(
    X_train,
    y_train,
    X_test,
    y_test,
    outcome_name,
    model_params
):

    final_rf = RandomForestClassifier(
        **model_params,
        random_state=42,
        n_jobs=-1
    )

    final_model = Pipeline([
        ("rf", final_rf)
    ])

    final_model.fit(X_train, y_train)

    y_test_pred = final_model.predict(X_test)
    y_test_prob = final_model.predict_proba(X_test)[:, 1]

    test_summary = pd.DataFrame({
        "Metric": [
            "AUC",
            "Precision",
            "Recall",
            "Accuracy",
            "F1"
        ],
        "Test Score": [
            roc_auc_score(y_test, y_test_prob),
            precision_score(
                y_test,
                y_test_pred,
                zero_division=0
            ),
            recall_score(
                y_test,
                y_test_pred,
                zero_division=0
            ),
            accuracy_score(
                y_test,
                y_test_pred
            ),
            f1_score(
                y_test,
                y_test_pred,
                zero_division=0
            )
        ]
    })

    test_summary["Test Score"] = test_summary[
        "Test Score"
    ].round(3)

    print(f"\n{outcome_name}: Independent test performance")
    print(test_summary.to_string(index=False))

    return {
        "model": final_model,
        "test_summary": test_summary
    }


In [17]:
# 9. Feasibility RF test evaluation
rf_feasibility_test_results = evaluate_rf_test(
    X_train=X_train,
    y_train=y_train_feasibility,
    X_test=X_test,
    y_test=y_test_feasibility,
    outcome_name="Feasibility Random Forest",
    model_params=feasibility_rf_results["best_params"]
)


Feasibility Random Forest: Independent test performance
   Metric  Test Score
      AUC       0.625
Precision       0.773
   Recall       0.994
 Accuracy       0.771
       F1       0.870


In [18]:
# Save results
joblib.dump(
    rf_feasibility_test_results,
    "../results/rf_feasibility_test_results.joblib"
)

rf_feasibility_test_results["test_summary"].to_csv(
    "../results/rf_feasibility_test_summary.csv",
    index=False
)

In [19]:
# 10. Willingness RF test evaluation
rf_willingness_test_results = evaluate_rf_test(
    X_train=X_train,
    y_train=y_train_willingness,
    X_test=X_test,
    y_test=y_test_willingness,
    outcome_name="Willingness Random Forest",
    model_params=willingness_rf_results["best_params"]
)


Willingness Random Forest: Independent test performance
   Metric  Test Score
      AUC       0.688
Precision       0.855
   Recall       0.993
 Accuracy       0.851
       F1       0.919


In [20]:
# Save results
joblib.dump(
    rf_willingness_test_results,
    "../results/rf_willingness_test_results.joblib"
)

# Save test performance table
rf_willingness_test_results["test_summary"].to_csv(
    "../results/rf_willingness_test_summary.csv",
    index=False
)